In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
from collections import Counter

In [ ]:
root = "/content/drive/MyDrive/CellMob_AUB/CellMob"

In [ ]:
print(os.path.exists(root))
print(os.listdir(root)[:10])

True
['bus_jeddah', 'bus_mekkah', 'bus_colored_kaust', 'car_jeddah', 'car_kz', 'car_mekkah', 'car_kaust', 'train_mekkah', 'walk_jeddah', 'walk_mekkah']


In [ ]:
rows = []

for folder in os.listdir(root):
    folder_path = os.path.join(root, folder)
    if not os.path.isdir(folder_path):
        continue

    parts = folder.split("_")
    label = parts[0]
    city = parts[-1]

    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            rows.append({
                "folder": folder,
                "file": file,
                "city": city,
                "label": label,
                "path": os.path.join(folder_path, file)
            })

df_index = pd.DataFrame(rows)

print("Number of CSV files:", len(df_index))
print("\nCities:", sorted(df_index["city"].unique()))
print("\nLabels:", sorted(df_index["label"].unique()))
print("\nFiles per city:")
print(df_index["city"].value_counts())

print("\nFiles per label:")
print(df_index["label"].value_counts())

df_index.head()

Number of CSV files: 160

Cities: ['jeddah', 'kaust', 'kz', 'mekkah']

Labels: ['bus', 'car', 'train', 'walk']

Files per city:
city
kaust     84
mekkah    46
jeddah    21
kz         9
Name: count, dtype: int64

Files per label:
label
walk     92
car      38
bus      28
train     2
Name: count, dtype: int64


,folder,file,city,label,path
0,bus_jeddah,bus_jeddah_011.csv,jeddah,bus,/content/drive/MyDrive/CellMob_AUB/CellMob/bus...
1,bus_jeddah,bus_jeddah_010.csv,jeddah,bus,/content/drive/MyDrive/CellMob_AUB/CellMob/bus...
2,bus_jeddah,bus_jeddah_009.csv,jeddah,bus,/content/drive/MyDrive/CellMob_AUB/CellMob/bus...
3,bus_jeddah,bus_jeddah_008.csv,jeddah,bus,/content/drive/MyDrive/CellMob_AUB/CellMob/bus...
4,bus_mekkah,bus_mekkah_023.csv,mekkah,bus,/content/drive/MyDrive/CellMob_AUB/CellMob/bus...


In [ ]:
column_counter = Counter()
file_column_counts = []

for path in df_index["path"]:
    try:
        df_tmp = pd.read_csv(path, nrows=1, low_memory=False)
        cols = list(df_tmp.columns)

        file_column_counts.append({
            "path": path,
            "n_columns": len(cols)
        })

        for c in cols:
            column_counter[c] += 1

    except Exception as e:
        print(f"Error reading {path}: {e}")

df_columns = pd.DataFrame({
    "column": list(column_counter.keys()),
    "appears_in_n_files": list(column_counter.values())
}).sort_values(by=["appears_in_n_files","column"], ascending=[False,True])

df_file_cols = pd.DataFrame(file_column_counts)

print("Total unique columns across all files:", len(df_columns))
print("\nDifferent file widths found:")
print(df_file_cols["n_columns"].value_counts().sort_index())

all_file_cols = df_columns[df_columns["appears_in_n_files"] == len(df_index)]

print("\nNumber of columns appearing in all files:", len(all_file_cols))

Total unique columns across all files: 91

Different file widths found:
n_columns
54    10
60     9
66    30
72    26
78    51
79     9
84    10
90    15
Name: count, dtype: int64

Number of columns appearing in all files: 54


In [ ]:
common_columns = all_file_cols["column"].tolist()

clean_dfs = []

for _, row in df_index.iterrows():
    try:
        df = pd.read_csv(row["path"], low_memory=False)

        # keep only columns that exist in every file
        df = df[common_columns].copy()

        # add metadata
        df["city"] = row["city"]
        df["label"] = row["label"]
        df["source_file"] = row["file"]

        clean_dfs.append(df)

    except Exception as e:
        print(f"Error processing {row['path']}: {e}")

df_all = pd.concat(clean_dfs, ignore_index=True)

print("Combined shape:", df_all.shape)
print("\nColumns:")
print(df_all.columns.tolist())

print("\nCities in combined data:")
print(df_all["city"].value_counts())

print("\nLabels in combined data:")
print(df_all["label"].value_counts())

Combined shape: (4083476, 57)

Columns:
['CQI for codeword 0 per subband - 1', 'CQI for codeword 0 per subband - 10', 'CQI for codeword 0 per subband - 11', 'CQI for codeword 0 per subband - 12', 'CQI for codeword 0 per subband - 13', 'CQI for codeword 0 per subband - 2', 'CQI for codeword 0 per subband - 3', 'CQI for codeword 0 per subband - 4', 'CQI for codeword 0 per subband - 5', 'CQI for codeword 0 per subband - 6', 'CQI for codeword 0 per subband - 7', 'CQI for codeword 0 per subband - 8', 'CQI for codeword 0 per subband - 9', 'CQI for codeword 1 per subband - 1', 'CQI for codeword 1 per subband - 10', 'CQI for codeword 1 per subband - 11', 'CQI for codeword 1 per subband - 12', 'CQI for codeword 1 per subband - 13', 'CQI for codeword 1 per subband - 2', 'CQI for codeword 1 per subband - 3', 'CQI for codeword 1 per subband - 4', 'CQI for codeword 1 per subband - 5', 'CQI for codeword 1 per subband - 6', 'CQI for codeword 1 per subband - 7', 'CQI for codeword 1 per subband - 8', '

In [ ]:
feature_cols = [c for c in df_all.columns if c not in ["city", "label", "source_file"]]

# convert everything to numeric
for col in feature_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

# fill missing values
df_all[feature_cols] = df_all[feature_cols].fillna(0)

print("Remaining NaNs:", df_all[feature_cols].isna().sum().sum())

Remaining NaNs: 0


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_all["label_encoded"] = le.fit_transform(df_all["label"])

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'bus': np.int64(0), 'car': np.int64(1), 'train': np.int64(2), 'walk': np.int64(3)}


In [ ]:
df_sample = df_all.sample(n=200000, random_state=42)

print(df_sample.shape)

(200000, 58)


In [ ]:
X = df_sample[feature_cols]
y = df_sample["label_encoded"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.86      0.88      7425
           1       0.93      0.87      0.90     11436
           2       0.98      0.97      0.97       938
           3       0.93      0.98      0.95     20201

    accuracy                           0.92     40000
   macro avg       0.93      0.92      0.93     40000
weighted avg       0.92      0.92      0.92     40000



In [ ]:
client_datasets = {}

for city in df_all["city"].unique():
    df_city = df_all[df_all["city"] == city]

    X_city = df_city[feature_cols].values
    y_city = df_city["label_encoded"].values

    client_datasets[city] = (X_city, y_city)

for city in client_datasets:
    print(city, client_datasets[city][0].shape)

jeddah (903774, 54)
mekkah (1191474, 54)
kaust (1832613, 54)
kz (155615, 54)


In [ ]:
from sklearn.linear_model import LogisticRegression

global_model = LogisticRegression(
    max_iter=200,
    n_jobs=-1,
    multi_class="multinomial"
)

In [ ]:
import numpy as np

client_samples = {}
max_per_client = 100000

for city in df_all["city"].unique():
    df_city = df_all[df_all["city"] == city]

    if len(df_city) > max_per_client:
        df_city = df_city.sample(n=max_per_client, random_state=42)

    X_city = df_city[feature_cols].values.astype(np.float32)
    y_city = df_city["label_encoded"].values.astype(np.int64)

    client_samples[city] = (X_city, y_city)

for city in client_samples:
    print(city, client_samples[city][0].shape, np.unique(client_samples[city][1]))

jeddah (100000, 54) [0 1 3]
mekkah (100000, 54) [0 1 2 3]
kaust (100000, 54) [0 1 3]
kz (100000, 54) [1 3]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
class TMDNet(nn.Module):
    def __init__(self, input_dim=54, hidden_dim=64, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
batch_size = 256
client_loaders = {}

for city, (X_city, y_city) in client_samples.items():
    X_tensor = torch.tensor(X_city, dtype=torch.float32)
    y_tensor = torch.tensor(y_city, dtype=torch.long)

    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    client_loaders[city] = loader

for city in client_loaders:
    print(city, len(client_loaders[city]))

jeddah 391
mekkah 391
kaust 391
kz 391


In [ ]:
df_test = df_all.sample(n=40000, random_state=42)

X_test_fl = torch.tensor(df_test[feature_cols].values.astype(np.float32), dtype=torch.float32)
y_test_fl = df_test["label_encoded"].values

In [ ]:
def train_local(model, loader, epochs=1, lr=0.001):
    model = model.to(device)
    model.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for _ in range(epochs):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    return model.state_dict()


def fedavg_aggregate(local_states, client_sizes):
    new_state = deepcopy(local_states[0])
    total_size = sum(client_sizes)

    for key in new_state.keys():
        new_state[key] = sum(
            (client_sizes[i] / total_size) * local_states[i][key]
            for i in range(len(local_states))
        )

    return new_state


def evaluate_model(model, X_test, y_test):
    model = model.to(device)
    model.eval()

    with torch.no_grad():
        outputs = model(X_test.to(device))
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    print("Accuracy:", acc)
    print(classification_report(y_test, preds))

    return acc, preds

In [ ]:
global_model = TMDNet(input_dim=len(feature_cols), hidden_dim=64, num_classes=4).to(device)

num_rounds = 5
local_epochs = 1

for rnd in range(num_rounds):
    local_states = []
    client_sizes = []

    for city, loader in client_loaders.items():
        local_model = deepcopy(global_model)
        local_state = train_local(local_model, loader, epochs=local_epochs, lr=0.001)

        local_states.append(deepcopy(local_state))
        client_sizes.append(len(loader.dataset))

    new_global_state = fedavg_aggregate(local_states, client_sizes)
    global_model.load_state_dict(new_global_state)

    print(f"Completed FedAvg round {rnd+1}/{num_rounds}")

    acc, _ = evaluate_model(global_model, X_test_fl, y_test_fl)

Completed FedAvg round 1/5
Accuracy: 0.430525
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      7381
           1       0.32      0.54      0.40     11414
           2       0.00      0.00      0.00       948
           3       0.53      0.55      0.54     20257

    accuracy                           0.43     40000
   macro avg       0.21      0.27      0.24     40000
weighted avg       0.36      0.43      0.39     40000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Completed FedAvg round 2/5
Accuracy: 0.475875
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      7381
           1       0.35      0.39      0.37     11414
           2       0.00      0.00      0.00       948
           3       0.54      0.72      0.61     20257

    accuracy                           0.48     40000
   macro avg       0.22      0.28      0.25     40000
weighted avg       0.37      0.48      0.42     40000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Completed FedAvg round 3/5
Accuracy: 0.4871
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      7381
           1       0.37      0.45      0.41     11414
           2       0.00      0.00      0.00       948
           3       0.55      0.71      0.62     20257

    accuracy                           0.49     40000
   macro avg       0.23      0.29      0.26     40000
weighted avg       0.38      0.49      0.43     40000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Completed FedAvg round 4/5
Accuracy: 0.5123
              precision    recall  f1-score   support

           0       0.50      0.00      0.00      7381
           1       0.42      0.37      0.39     11414
           2       0.00      0.00      0.00       948
           3       0.54      0.80      0.65     20257

    accuracy                           0.51     40000
   macro avg       0.37      0.29      0.26     40000
weighted avg       0.49      0.51      0.44     40000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Completed FedAvg round 5/5
Accuracy: 0.483175
              precision    recall  f1-score   support

           0       0.67      0.00      0.00      7381
           1       0.37      0.45      0.41     11414
           2       0.00      0.00      0.00       948
           3       0.54      0.70      0.61     20257

    accuracy                           0.48     40000
   macro avg       0.39      0.29      0.25     40000
weighted avg       0.50      0.48      0.43     40000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
def train_local_fedavg_matched(model, loader, epochs=1, lr=0.001):  ##################################test
    model = model.to(device)
    model.train()

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

    return model.state_dict()

In [ ]:
global_model_matched = TMDNet(input_dim=len(feature_cols), hidden_dim=64, num_classes=4).to(device)

num_rounds = 5
local_epochs = 1
fedavg_matched_accuracies = []

for rnd in range(num_rounds):
    local_states = []
    client_sizes = []

    for city, loader in client_loaders.items():
        local_model = deepcopy(global_model_matched)
        local_state = train_local_fedavg_matched(local_model, loader, epochs=local_epochs, lr=0.001)

        local_states.append(deepcopy(local_state))
        client_sizes.append(len(loader.dataset))

    new_global_state = fedavg_aggregate(local_states, client_sizes)
    global_model_matched.load_state_dict(new_global_state)

    print(f"\nCompleted matched FedAvg round {rnd+1}/{num_rounds}")
    acc, _ = evaluate_model(global_model_matched, X_test_fl, y_test_fl)
    fedavg_matched_accuracies.append(acc)


Completed matched FedAvg round 1/5
Accuracy: 0.327925
              precision    recall  f1-score   support

           0       0.29      0.05      0.09      7381
           1       0.28      0.72      0.40     11414
           2       1.00      0.01      0.01       948
           3       0.50      0.22      0.31     20257

    accuracy                           0.33     40000
   macro avg       0.52      0.25      0.20     40000
weighted avg       0.41      0.33      0.29     40000


Completed matched FedAvg round 2/5
Accuracy: 0.467175
              precision    recall  f1-score   support

           0       0.26      0.04      0.06      7381
           1       0.32      0.26      0.29     11414
           2       0.18      0.01      0.02       948
           3       0.52      0.76      0.62     20257

    accuracy                           0.47     40000
   macro avg       0.32      0.27      0.25     40000
weighted avg       0.41      0.47      0.41     40000


Completed matched F

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Completed matched FedAvg round 4/5
Accuracy: 0.462875
              precision    recall  f1-score   support

           0       0.19      0.01      0.03      7381
           1       0.33      0.34      0.34     11414
           2       0.00      0.00      0.00       948
           3       0.52      0.72      0.60     20257

    accuracy                           0.46     40000
   macro avg       0.26      0.27      0.24     40000
weighted avg       0.39      0.46      0.41     40000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Completed matched FedAvg round 5/5
Accuracy: 0.49765
              precision    recall  f1-score   support

           0       0.21      0.02      0.03      7381
           1       0.37      0.16      0.23     11414
           2       0.00      0.00      0.00       948
           3       0.52      0.88      0.66     20257

    accuracy                           0.50     40000
   macro avg       0.27      0.27      0.23     40000
weighted avg       0.41      0.50      0.40     40000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

def evaluate_model(model, X_test, y_test):
    model = model.to(device)
    model.eval()

    with torch.no_grad():
        outputs = model(X_test.to(device))
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    print("Accuracy:", acc)
    print(classification_report(y_test, preds, zero_division=0))
    return acc, preds

In [ ]:
import pandas as pd
import torch

all_y = []
for city, (X_city, y_city) in client_samples.items():
    all_y.extend(y_city.tolist())

class_counts = pd.Series(all_y).value_counts().sort_index()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_weights)

class_weights_tensor = torch.tensor(class_weights.values, dtype=torch.float32).to(device)
print(class_counts)
print(class_weights_tensor)

0     64705
1    182517
2      7875
3    144903
Name: count, dtype: int64
tensor([0.3993, 0.1416, 3.2808, 0.1783])


In [ ]:
from copy import deepcopy
import torch
import torch.nn as nn
import torch.optim as optim

def zeros_like_state_dict(state_dict):
    return {k: torch.zeros_like(v, device=device) for k, v in state_dict.items()}

def move_state_to_device(state_dict, device):
    return {k: v.clone().detach().to(device) for k, v in state_dict.items()}

def scaffold_train_local(model, loader, c_global, c_local, epochs=1, lr=0.01):
    model = deepcopy(model).to(device)
    model.train()

    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.SGD(model.parameters(), lr=lr)

    global_state = move_state_to_device(model.state_dict(), device)

    step_count = 0

    for _ in range(epochs):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()

            with torch.no_grad():
                for name, param in model.named_parameters():
                    if param.grad is not None:
                        param.grad += (c_global[name] - c_local[name])
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            step_count += 1

    local_state = move_state_to_device(model.state_dict(), device)

    new_c_local = {}
    for name in local_state:
        new_c_local[name] = c_local[name] - c_global[name] + (global_state[name] - local_state[name]) / step_count

    delta_c = {}
    for name in new_c_local:
        delta_c[name] = new_c_local[name] - c_local[name]

    return local_state, new_c_local, delta_c

In [ ]:
def average_model_states(local_states, client_sizes):
    total_size = sum(client_sizes)
    new_state = {}

    for key in local_states[0].keys():
        new_state[key] = sum(
            (client_sizes[i] / total_size) * local_states[i][key]
            for i in range(len(local_states))
        )

    return new_state

def update_server_control(c_global, delta_cs, beta=0.25):
    num_clients = len(delta_cs)
    new_c_global = {}

    for key in c_global.keys():
        avg_delta = sum(delta_c[key] for delta_c in delta_cs) / num_clients
        new_c_global[key] = c_global[key] + beta * avg_delta

    return new_c_global

In [ ]:
scaffold_model = TMDNet(input_dim=len(feature_cols), hidden_dim=64, num_classes=4).to(device)
scaffold_model.load_state_dict(deepcopy(global_model_matched.state_dict()))

c_global = update_server_control(c_global, delta_cs, beta=0.25)
c_locals = {city: zeros_like_state_dict(scaffold_model.state_dict()) for city in client_loaders.keys()}

print("Fresh SCAFFOLD initialized.")

Fresh SCAFFOLD initialized.


In [ ]:
num_rounds_scaffold = 10
local_epochs_scaffold = 1
lr_scaffold = 0.0005

scaffold_accuracies = []

for rnd in range(num_rounds_scaffold):
    local_states = []
    client_sizes = []
    delta_cs = []

    for city, loader in client_loaders.items():
        local_state, new_c_local, delta_c = scaffold_train_local(
            scaffold_model,
            loader,
            c_global,
            c_locals[city],
            epochs=local_epochs_scaffold,
            lr=lr_scaffold
        )

        local_states.append(local_state)
        client_sizes.append(len(loader.dataset))
        c_locals[city] = new_c_local
        delta_cs.append(delta_c)

    new_global_state = average_model_states(local_states, client_sizes)
    scaffold_model.load_state_dict(new_global_state)

    c_global = update_server_control(c_global, delta_cs)

    print(f"\nCompleted corrected SCAFFOLD round {rnd+1}/{num_rounds_scaffold}")
    acc, _ = evaluate_model(scaffold_model, X_test_fl, y_test_fl)
    scaffold_accuracies.append(acc)


Completed corrected SCAFFOLD round 1/10
Accuracy: 0.4933
              precision    recall  f1-score   support

           0       0.26      0.02      0.05      7381
           1       0.39      0.36      0.38     11414
           2       0.00      0.00      0.00       948
           3       0.54      0.76      0.63     20257

    accuracy                           0.49     40000
   macro avg       0.30      0.29      0.26     40000
weighted avg       0.43      0.49      0.43     40000


Completed corrected SCAFFOLD round 2/10
Accuracy: 0.492375
              precision    recall  f1-score   support

           0       0.25      0.02      0.04      7381
           1       0.39      0.35      0.37     11414
           2       0.00      0.00      0.00       948
           3       0.53      0.77      0.63     20257

    accuracy                           0.49     40000
   macro avg       0.29      0.28      0.26     40000
weighted avg       0.43      0.49      0.43     40000


Completed c